In [1]:
import os

import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ["base_url"]


model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

# 消息格式


## Tuple - ChatPromptTemplate

**说明**：在 `ChatPromptTemplate` 中，Tuple 的第一个元素是角色字符串（如 `"system"`、`"human"`、`"ai"`），第二个元素是包含占位符的模板字符串。调用 `invoke` 时传入一个字典来填充这些占位符

("角色", "内容")

In [6]:
from langchain_core.prompts import ChatPromptTemplate

tuple_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个专业的翻译助手，请将以下内容翻译成{language}。"),
        ("human", "{text}"),
    ]
)

tuple_prompt

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='你是一个专业的翻译助手，请将以下内容翻译成{language}。'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [11]:
message_tuple = tuple_prompt.invoke({"language": "英语", "text": "我爱你"})
message_tuple

ChatPromptValue(messages=[SystemMessage(content='你是一个专业的翻译助手，请将以下内容翻译成英语。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我爱你', additional_kwargs={}, response_metadata={})])

In [13]:
model.invoke(message_tuple).pretty_print()

================================== Ai Message ==================================

I love you.


In [12]:
model.invoke(
    [
        ("system", "你是一个专业的翻译助手，请将以下内容翻译成英语。"),
        ("human", "我爱你"),
    ]
).pretty_print()

================================== Ai Message ==================================

I love you.


## 直接字符串

In [15]:
model.invoke("Hello World").pretty_print()

================================== Ai Message ==================================

Hello! How can I help you today?


## dict形式

In [ ]:
messages_dict = [
    {"role": "system", "content": "你是一个简洁的助手，回答不超过20字。"},
    {"role": "user", "content": "LangChain是什么？"},
    {"role": "assistant", "content": "一个用于构建LLM应用的框架。"},
    {"role": "user", "content": "它支持哪些语言？"},
]

In [17]:
model.invoke(messages_dict).pretty_print()

================================== Ai Message ==================================

主要支持Python和JS/TS。


## Message对象

In [19]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages_obj = [
    SystemMessage(content="你是一个精通Python的代码审查员。"),
    HumanMessage(content="请检查这段代码：print('hello')"),
    AIMessage(content="这段代码语法正确，但建议使用双引号以符合PEP8规范。"),
    HumanMessage(content="那变量命名呢？"),
]

In [21]:
model.invoke(messages_obj).pretty_print()

================================== Ai Message ==================================

这段代码里 **没有变量**，所以不存在“变量命名”可检查的地方。

如果打算改写成带变量的形式，命名建议如下：

```python
greeting = "hello"
print(greeting)
```

要点：

- 用 `snake_case`：如 `user_name`、`total_count`
- 见名知意：避免 `a`、`x`、`data1` 这类含义不清的名字
- 避免用 Python 内置名或关键字：如 `list`、`dict`、`type`、`class`
- 常量可用全大写：如 `MAX_RETRY = 3`
- 临时变量、循环索引可以用短名：如 `i`、`j`、`tmp`

另外补充一点：PEP8 并没有强制字符串必须用双引号，只建议同一项目中保持一致。所以 `print('hello')` 本身在变量命名和字符串引号上都没有明显问题。


# 流式输出

In [ ]:
response = model.stream(messages_obj)

for chunk in response:
    print(chunk.content, end="")

# 异步调用

In [25]:
r = model.ainvoke("Hello World")
r

<coroutine object BaseChatModel.ainvoke at 0x000001F2636F4040>

In [26]:
rsync = await r

In [27]:
rsync

AIMessage(content='Hello, World! 👋 How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 32, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 117, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 32}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a14f0819045b105e148abba05b4b58', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a14f08-3b63-79c1-8522-59f54f22c753-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 130, 'total_tokens': 162, 'input_token_details': {'cache_read': 0}, 'output_token_details'

# 提示词模版

1. ChatPromptTemplate
3. PromptTemplate

In [ ]:
"""
tuple_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个专业的翻译助手，请将以下内容翻译成{language}。"),
        ("human", "{text}"),
    ]
)

message_tuple = tuple_prompt.invoke({"language": "英语", "text": "我爱你"})
"""
message_tuple.to_messages()


[SystemMessage(content='你是一个专业的翻译助手，请将以下内容翻译成英语。', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='我爱你', additional_kwargs={}, response_metadata={})]

In [30]:
tuple_prompt.input_variables

['language', 'text']

In [34]:
from langchain_core.prompts import PromptTemplate

# prompt_template = PromptTemplate(
#     template="讲一个关于{topic}的冷笑话", input_variables=["topic"]
# )

# 能自己推断
prompt_template = PromptTemplate(template="讲一个关于{topic}的冷笑话")

In [35]:
prompt_template.input_variables

['topic']

In [36]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt_template | model | parser

In [ ]:
response = chain.invoke({"topic": "鼠标"})

In [39]:
print(response)

鼠标为什么总是冷冰冰的？

因为它天天被人点来点去，心早就凉了。
